# 01 - EDA: прогнозирование оттока абонентов телеком-оператора

Цель ноутбука: проверить датасет, найти target `Churn`, оценить пропуски, дубликаты, баланс классов и базовые связи признаков с оттоком.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
project_root = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data.loader import (
    FEATURE_DESCRIPTIONS,
    TARGET_COLUMN,
    dataset_quality_summary,
    find_dataset,
    load_churn_data,
)

sns.set_theme(style='whitegrid')

## 1. Поиск и загрузка датасета

In [ ]:
dataset_path = find_dataset()
summary = dataset_quality_summary(dataset_path)
df = load_churn_data(dataset_path, remove_duplicates=True)

print('Использованный файл:', dataset_path)
print('Raw shape:', (summary['raw_rows'], summary['raw_columns']))
print('Clean shape:', df.shape)
summary

## 2. Типы данных, пропуски и дубликаты

In [ ]:
display(df.head())
display(df.dtypes.rename('dtype').to_frame())
print('Пропуски после очистки:', int(df.isna().sum().sum()))
print('Дубликаты после очистки:', int(df.duplicated().sum()))

## 3. Целевая переменная и дисбаланс

In [ ]:
target_counts = df[TARGET_COLUMN].value_counts().sort_index()
target_share = df[TARGET_COLUMN].value_counts(normalize=True).sort_index()
display(pd.DataFrame({'count': target_counts, 'share': target_share}))

ax = sns.countplot(data=df, x=TARGET_COLUMN, hue=TARGET_COLUMN, palette='Set2')
ax.set_title('Распределение Churn')
ax.set_xlabel('Churn')
ax.set_ylabel('Количество')
ax.legend_.remove()
plt.show()

## 4. Описание признаков

In [ ]:
display(pd.DataFrame(
    [{'feature': key, 'description': value} for key, value in FEATURE_DESCRIPTIONS.items()]
))

## 5. Корреляции

In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(11, 9))
sns.heatmap(corr, cmap='coolwarm', center=0, square=True)
plt.title('Корреляционная матрица')
plt.show()

display(corr[TARGET_COLUMN].drop(TARGET_COLUMN).sort_values(key=lambda s: s.abs(), ascending=False).to_frame('corr_with_churn'))

## 6. Распределения ключевых числовых признаков

In [ ]:
numeric_features = [
    'call_failures', 'subscription_length', 'seconds_of_use',
    'frequency_of_use', 'frequency_of_sms', 'distinct_called_numbers',
    'customer_value'
]
df[numeric_features].hist(figsize=(12, 9), bins=30)
plt.suptitle('Распределения числовых признаков')
plt.tight_layout()
plt.show()

## 7. Сравнение признаков по классам Churn

In [ ]:
compare_features = [
    'complaints', 'status', 'frequency_of_use',
    'seconds_of_use', 'customer_value', 'distinct_called_numbers'
]
melted = df.melt(id_vars=TARGET_COLUMN, value_vars=compare_features, var_name='feature', value_name='value')
grid = sns.catplot(
    data=melted,
    x=TARGET_COLUMN,
    y='value',
    col='feature',
    kind='box',
    col_wrap=3,
    sharey=False,
    height=3.2,
)
grid.fig.suptitle('Признаки по классам Churn', y=1.03)
plt.show()

display(df.groupby(TARGET_COLUMN)[compare_features].mean().T)

## Краткий вывод

Данные без пропусков, но с 300 exact-дубликатами. После удаления дублей остаётся дисбаланс: класс оттока около 15.65%. Для оценки моделей нельзя ориентироваться только на accuracy; нужны Recall, F1 и ROC-AUC.